# 🌍 GeoPredict — Global Pulse Sentiment & Alert Pipeline
### Powered by Fine-Tuned RoBERTa Transformer & Market Relevance Filtering

This notebook orchestrates the news intelligence pipeline:
1. **News Ingestion**: Loads global news headlines and descriptions (`news_data.csv`).
2. **RoBERTa Sentiment Analysis**: Replaces legacy NLTK VADER with a fine-tuned RoBERTa model specialized in financial/economic sentiment.
3. **Market Relevance Filtering**: Identifies high-impact market-moving geopolitical developments.
4. **Export & Alerts**: Formats and saves alerts to `rsentiments.csv` and `sentiment.csv`.
5. **Model Validation**: Displays test set evaluation metrics on the held-out benchmark.


In [27]:
# Step 1: Environment & Project Setup
import os
import sys
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv

NOTEBOOK_DIR = Path.cwd() if Path.cwd().name == "notebooks" else Path("notebooks").resolve()
REPO_ROOT = NOTEBOOK_DIR.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

load_dotenv(NOTEBOOK_DIR / ".env")
print(f"Project root resolved: {REPO_ROOT}")


Project root resolved: /home/blackeye/Projects/Geo_Predict


In [28]:
# Step 2: News Data Ingestion
news_csv_path = REPO_ROOT / "WEB_SCRAPED" / "news_data.csv"

try:
    df_news = pd.read_csv(news_csv_path)
    print(f"Loaded {len(df_news)} news articles from {news_csv_path}")
    display(df_news.head(3))
except FileNotFoundError:
    print(f"File {news_csv_path} not found. Please verify data collection.")


Loaded 31 news articles from /home/blackeye/Projects/Geo_Predict/WEB_SCRAPED/news_data.csv


,Headline,Description,Time,Category,SourceFile
0,Record number of purges as Xi emphasises 'part...,"Xi's anti-corruption campaign has become ""an e...",5 hrs ago,Asia,article_17_news.html
1,'I live with a ghost' - inside the haunted hom...,Properties where someone has died in tragic ci...,8 hrs ago,Business,article_2_business.html
2,Hello Kitty maker cuts ties with Japanese spy ...,Sanrio has apologised after its merchandise li...,10 hrs ago,Asia,article_1_business.html


In [29]:
# Step 3: RoBERTa Sentiment Analysis (Replaces NLTK VADER)
from models.roberta_sentiment import RobertaSentimentAnalyzer, analyze_sentiment

# Initialize RoBERTa sentiment analyzer (loads fine-tuned weights from models/roberta_finetuned)
analyzer = RobertaSentimentAnalyzer()

# Combine headline and description for contextual sentiment analysis
headlines_to_analyze = (df_news["Headline"].fillna("") + ". " + df_news["Description"].fillna("")).tolist()

sentiment_results = analyze_sentiment(headlines_to_analyze, analyzer=analyzer)

# Attach sentiment predictions to DataFrame
df_news["sentiment_label"] = [item["label"] for item in sentiment_results]
df_news["sentiment_score"] = [item["sentiment_score"] for item in sentiment_results]
df_news["sentiment_confidence"] = [item["confidence"] for item in sentiment_results]

print(f"RoBERTa Sentiment Analysis complete for {len(df_news)} articles.")
print("\n--- Top Bullish Events ---")
display(df_news[df_news["sentiment_label"] == "Positive"][["Headline", "sentiment_label", "sentiment_score"]].head(3))

print("\n--- Top Bearish Events ---")
display(df_news[df_news["sentiment_label"] == "Negative"][["Headline", "sentiment_label", "sentiment_score"]].head(3))


Loading RoBERTa Sentiment Model from '/home/blackeye/Projects/Geo_Predict/models/roberta_finetuned' on device 'cuda'...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 4652.01it/s]


RoBERTa Sentiment Analysis complete for 31 articles.

--- Top Bullish Events ---


,Headline,sentiment_label,sentiment_score
3,"Building boom puts 300,000 people in the path ...",Positive,0.9673
4,"Elon Musk is a 'real genius', Putin says after...",Positive,0.9669
6,Careless dismissals give Australia route to se...,Positive,0.9615



--- Top Bearish Events ---


,Headline,sentiment_label,sentiment_score
0,Record number of purges as Xi emphasises 'part...,Negative,-0.0888
2,Hello Kitty maker cuts ties with Japanese spy ...,Negative,-0.9347
8,Hong Kong's leader apologises for deadly high-...,Negative,-0.7713


In [31]:
# Step 5: RoBERTa Test Set Validation Summary
import json

metrics_file = REPO_ROOT / "models" / "roberta_finetuned" / "test_results.json"
if metrics_file.exists():
    with open(metrics_file, "r") as f:
        metrics = json.load(f)
    print("=" * 55)
    print("      RoBERTa Model Validation on Held-Out Test Set")
    print("=" * 55)
    print(f"• Test Accuracy:        {metrics.get('test_accuracy', 0):.4f} ({metrics.get('test_accuracy', 0)*100:.2f}%)")
    print(f"• Test Macro F1:        {metrics.get('test_macro_f1', 0):.4f}")
    print(f"• Test Weighted F1:     {metrics.get('test_weighted_f1', 0):.4f}")
    print(f"• Test Macro Precision: {metrics.get('test_macro_precision', 0):.4f}")
    print(f"• Test Macro Recall:    {metrics.get('test_macro_recall', 0):.4f}")
    print(f"• Total Test Samples:   {metrics.get('total_test_samples', 0)}")
    print("=" * 55)
else:
    print("Note: Run `python models/train_sentiment.py` to evaluate and generate test set validation metrics.")


      RoBERTa Model Validation on Held-Out Test Set
• Test Accuracy:        0.9538 (95.38%)
• Test Macro F1:        0.9497
• Test Weighted F1:     0.9540
• Test Macro Precision: 0.9440
• Test Macro Recall:    0.9556
• Total Test Samples:   346
